# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

⠸ cffi==2.1.1                                                                   Resolved 76 packages in 599ms
Prepared 71 packages in 1.14s                                                gradio               ------------------------------ 28.34 MiB/29.93 MiB         gradio               ------------------------------ 3.42 MiB/29.93 MiB          gradio               ------------------------------ 3.06 MiB/29.93 MiB          gradio               ------------------------------ 1.93 MiB/29.93 MiB          gradio               ------------------------------ 1.81 MiB/29.93 MiB          gradio               ------------------------------ 1019.00 KiB/29.93 MiB       gradio               ------------------------------ 1002.67 KiB/29.93 MiB       gradio               ------------------------------ 541.42 KiB/3.73 MiB         gradio               ------------------------------ 416.00 KiB/29.93 MiB        Agradio               ------------------------------ 272.00 KiB/29.93 MiB        radio       

## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr

/Users/kobe/Senior/IT3025C - Introduction to Agentic AI/Week 7/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [5]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-08-01-preview",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [11]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
NOTES_PATH = 'notes/notes.pdf'

def load_notes(path):
    try:
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        return text.strip()
    except FileNotFoundError:
        return "(notes.pdf not found yet: add it to this folder to enable real study content.)"

study_notes = load_notes(NOTES_PATH)

persona = ("Coliver Assistant, a collaborative assistant that provides constructive feedback, supports learning, and gives helpful hints when users answer incorrectly or are unsure how to respond.")
print(study_notes[:400])

Data Technology Study Notes
Topic: Core Concepts in Modern Data Systems
Databases and Data Models

Relational databases organize data into tables made of rows and columns. SQL is commonly used to
query and modify this data.

Primary keys uniquely identify rows, while foreign keys connect related tables.

NoSQL databases are useful when data is semi-structured, rapidly changing, or does not fit 


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [17]:
# TODO: your first tool, decorated with @function_tool
@function_tool
def bookmark_topic_tool(topic: str) -> str:
    # TODO: implement your tool. For example, print something and/or write it to a file.
    # Return a short string describing what happened.
    with open("review_topics.txt", "a") as file:
        file.write(f"Topic: {topic}\n")
    return f"Bookmarked topic: {topic}"


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [18]:
# TODO: your second tool, decorated with @function_tool
@function_tool
def simplify_topic_tool(topic: str) -> str:
    topic = " ".join(str(v) for v in value).strip()

    if not topic:
        return "No topic was provided."

    url = "https://en.wikipedia.org/w/rest.php/v1/search/page"

    params = {
        "q": topic,
        "limit": 1
    }

    headers = {
        "User-Agent": "StudyAssistant"
    }

    try:
        response = requests.get(url, params=params,headers=headers, timeout=5)
        response.raise_for_status()

        data = response.json()
        pages = data.get("pages", [])

        if not pages:
            return f"No information was found for '{topic}'."

        page = pages[0]

        title = page.get("title", topic)
        description = page.get("description", "")
        excerpt = page.get("excerpt", "")

        # Remove simple HTML tags from excerpt
        excerpt = (
            excerpt.replace("<span class=\"searchmatch\">", "")
            .replace("</span>", "")
        )

        return f"{title}: {description}. {excerpt}"

    except requests.exceptions.RequestException:
        return "The Wikipedia service is currently unavailable."

## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [19]:
# TODO: your third tool, decorated with @function_tool
@function_tool
def follow_up_question_tool(topic: str, level: str = "medium") -> str:
    topic = str(topic).strip()

    if not topic:
        return "No topic provided."

    return (
        f"Ask the student one follow-up question about '{topic}' "
        f"at {level} difficulty. Do not give the answer yet."
    )

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [20]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role
You are a collaborative study assistant.

Persona:
{persona}

# Context
Study notes:
{study_notes}

# Rules

When identifying a topic for a tool call, always use a short and accurate
topic name, such as "SQL vs NoSQL", "DNS", or "Python Lists".

If the student answers incorrectly, says they do not know,
or clearly struggles with a topic, call bookmark_topic_tool
to save that topic for future review.

Only bookmark the topic once per struggle.

If the student is confused, asks for a simpler explanation,
or continues to struggle, call simplify_topic_tool with the topic.

If the student answers correctly and additional practice would be useful,
call follow_up_question_tool with the same topic.

Use:
- "easy" if the student is struggling
- "medium" for normal practice
- "hard" if the student is doing well

Do not give the answer to a follow-up question immediately.
Wait for the student to answer first.
"""

study_assistant = Agent(
    name="Coliver Assistant",
    instructions=instructions,
    tools=[bookmark_topic_tool, simplify_topic_tool, follow_up_question_tool],
    model=azure_model,
)

print("Agent built:", study_assistant.name)


Agent built: Coliver Assistant


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [21]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_assistant, "What is SQL database?")
print(result.final_output)

A SQL database is a database that stores data in structured tables made of rows and columns, and it uses SQL, which stands for Structured Query Language, to read and change that data.

Example:
- A `Customers` table might store customer IDs, names, and emails.
- You can use SQL commands to:
  - get data: `SELECT`
  - add data: `INSERT`
  - change data: `UPDATE`
  - remove data: `DELETE`

SQL databases are usually relational databases, which means tables can be connected using keys like:
- primary key: uniquely identifies a row
- foreign key: links one table to another

Examples of SQL databases:
- MySQL
- PostgreSQL
- SQL Server
- SQLite

If you want, I can also explain the difference between a SQL database and a NoSQL database in one simple paragraph.


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [22]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(study_assistant, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.

## Reflection Answer
1. I copied the persona and notes unchanged from assignment 6. I also copied the tools from my previous assignments and added the decorator. After that, I also updated the format of the tool arguments.
2. I think the `Agent` and `Runner.run` replaced the role of looping through defining the system prompts and instructions the way we want the agent to behave. It also helps find what context the user has given and determine based on user's need which tools should be used and eventually use the correct tool and respond to user. Specificly, `Agent` defines the system prompts, model used, tools defined, and `Runner.run` is the one who loops through the context and decide which tools to be used.
3. I carried over from my assignment 6 into `instruction` directly
4. If I added a fourth tool to this agent, I would need to write a new function define with decorator and correct format of function argument. I also need to add that tool to `Agent`, and may need to tweak my instruction as well. I wouldn't need to change anything on the looping to find the context and define what tool to be used which Runner.run already handles that.